# Recreate image from text

- recreate image from text
- get representation of image in text

## Import modules

In [1]:
import itertools
from datetime import date
import pathlib
import cv2
import numpy as np
from numba import jit
import ray

## Recreate image from text & get representation of image in text

In [15]:
img = np.zeros((3,3,3), dtype=str)
img

array([[['', '', ''],
        ['', '', ''],
        ['', '', '']],

       [['', '', ''],
        ['', '', ''],
        ['', '', '']],

       [['', '', ''],
        ['', '', ''],
        ['', '', '']]], dtype='<U1')

In [16]:
text_img = np.empty_like(img, dtype=str)
text_img

array([[['', '', ''],
        ['', '', ''],
        ['', '', '']],

       [['', '', ''],
        ['', '', ''],
        ['', '', '']],

       [['', '', ''],
        ['', '', ''],
        ['', '', '']]], dtype='<U1')

In [115]:
# set project directory & directory where to save mosaic results images
project_dir = pathlib.Path('/Users/derrickvanfrausum/BeCode_AI/git-repos/coding-art/core/assets/images')
project_name = "mosaic_text"
mosaic_images_dir = project_dir / f"mosaic_results_{project_name}"
mosaic_images_dir.mkdir(parents=True, exist_ok=True)

# set list of texts to write
texts = "AB"
# texts = '¨^/|()[]@#-_%*;?.:+=!"$£{}€<>°`'
# texts = ["yo", "lo"]
# texts = "DP"
# texts = "たていすかんなにらせちとしはきくまのりれつさそひこみ"
# texts = ["kle", "boom", "mute", "your", "self"]

# set font
font = cv2.FONT_HERSHEY_SIMPLEX

# fontScale
fontScale = 2.0

# color in BGR
# color = (255, 255, 255)
color = (0, 0, 0)
# colors = [(0, 0, 255), (255, 0, 0), (0, 255, 0), (0, 0, 0), (255, 255, 255)]
# colors = [(255, 255, 255)]

# Line thickness of 2 px
thickness = 10

# choose number of tasks
nb_tasks = 4

# # split list into n chunks (n = number of multiprocessing tasks)
# img_paths_chunks = np.array_split(img_paths, nb_tasks)

# get photo to recreate
# photo_path = img_paths_chunks[0][0]
photo_path = "/Users/derrickvanfrausum/BeCode_AI/git-repos/learn-ai/opencv/assets/images/avatar_dd/db/avatar_dd.png"
print(photo_path)
photo = cv2.cvtColor(cv2.imread(photo_path), cv2.COLOR_BGR2GRAY)

# set grid caracteristics
number_rows = 20
number_cols = 20

# read and get first image shape as reference
ref_img = photo.copy()
ref_img_height, ref_img_width = ref_img.shape
ref_img_height, ref_img_width = ref_img_height//4, ref_img_width//4

# set output grid image shape
out_img_height = ref_img_height * number_rows
out_img_width = ref_img_width * number_cols

# set image matrix canvas & & text array
img_matrix = np.ones((out_img_height, out_img_width), np.uint8)*255
# img_matrix.fill(255)
text_np = np.empty((number_cols, number_rows), dtype='U256')
text_np.fill(" ")


# shutdown ray
ray.shutdown()

# start ray
# WARNING worker.py:462 -- The driver may not be able to keep up with the stdout/stderr of the workers. To avoid forwarding logs to the driver, use 'ray.init(log_to_driver=False)'.Ò
ray.init(log_to_driver=False)

# @jit(nopython=True)
@ray.remote
def create_mosaic_with_text(photo, texts, number_rows, number_cols, grid_range):
    """
    Function to recreate given photo with photos mosaic
    Arguments:
    * grid_range: to assign grid range for each worker
    """

    # set image matrix canvas 
    text_img = np.ones((ref_img_height, ref_img_width), np.uint8)*255
    # text_img.fill(255)
    

    photo = cv2.resize(photo, (out_img_width, out_img_height))

    # create empty dictionary to track grid positions and their corresponding image numpy arrays
    imgs_dict = {}

    # loop through grid cell positions
    grid_positions = itertools.product(range(*grid_range), range(number_rows))
    for (x_i, y_i) in grid_positions:
        print(x_i, y_i)

        # get region of interest in given photo to recreate
        x = x_i * ref_img_width
        y = y_i * ref_img_height
        roi = photo[y:y + ref_img_height, x:x + ref_img_width]

        # inititiate tracker for best match to that roi
        best_match = np.inf
        best_match_index = 0

        # get an image with required shape and that best match photo to recreate
        for img_idx, text in enumerate(texts):

            # copy img
            img = text_img.copy()

            # get boundary of this text
            text_size = cv2.getTextSize(text, font, fontScale, thickness)[0]

            # get coords based on boundary
            text_x = (img.shape[1] - text_size[0]) // 2
            text_y = (img.shape[0] + text_size[1]) // 2

            # add text
            img = cv2.putText(img, text, (text_x, text_y), font, fontScale,
            0, thickness, cv2.LINE_AA, False)

            # # get image shape
            # img_height, img_width, img_channel = img.shape

            # # get center of image
            # center_x, center_y = (img_height // 2, img_width // 2)

            # # if same image shape as ref, compare image with roi
            # if (img_height, img_width, img_channel)  == (ref_img_height, ref_img_width, ref_img_channel):
            total_sum = np.sum(np.abs(roi - img)) # toDO: try cv2.compareHist(H1, H2, method)
            if total_sum < best_match:
                best_match = total_sum
                best_match_index = img_idx

        best_match_text = texts[best_match_index]

        # copy img
        img = text_img.copy()

        # add text
        img = cv2.putText(img, best_match_text, (text_x, text_y), font, fontScale,
        0, thickness, cv2.LINE_AA, False)

        # append grid positions and image cell to dict
        imgs_dict[(best_match_text,y,x)] = img
        
    
    return imgs_dict


# start tasks in parallel
result_ids = []
# chunk_nb = 0
for i in range(0, number_cols, number_cols//nb_tasks): # set grid range for each worker

    # recreate given photo with photos mosaic
    result_ids.append(create_mosaic_with_text.remote(photo, texts, number_rows, number_cols, (i, i+number_cols//nb_tasks)))

    # # increment chunk number
    # chunk_nb += 1
    
# wait for the tasks to complete and retrieve the results
results = ray.get(result_ids)

out_text = ''

# combine results and save image matrix
for result in results:
    for key, cell_image in result.items():
        best_match_text, y, x = key
        img_matrix[y:y+ref_img_height, x:x+ref_img_width] = cell_image
        y = y//ref_img_height
        x = x//ref_img_width
        if x == text_np.shape[1] - 1:
            text_np[y, x] = best_match_text + '\n'
        else:
            text_np[y, x] = best_match_text

# get current date
today = date.today().strftime("%Y%m%d")

# set output image & metadata paths
out_img_path = str(mosaic_images_dir / f"matrix_photo_{number_rows*number_cols}_{today}.jpg")
out_metadata_path = str(mosaic_images_dir / f"best_match_texts_{number_rows*number_cols}_{today}.npy")

# save output image & metadata
cv2.imwrite(out_img_path, img_matrix)
np.save(out_metadata_path, text_np)

 # shutdown ray
ray.shutdown()

/Users/derrickvanfrausum/BeCode_AI/git-repos/learn-ai/opencv/assets/images/avatar_dd/db/avatar_dd.png


In [116]:
out_text = ""
grid_positions = itertools.product(range(number_rows), range(number_cols))
for (y, x) in grid_positions:
        out_text += text_np[y, x]   
print(out_text)

AAAAAAAAAAAAAAAAAAAA
AAAAAAAAAABBAAAAAAAA
AAAAAAABBBBABAAAAAAA
AAAAAAAABAABBBAAAAAA
AAAAAABBBABABBAAAAAA
AAAAAABBAAABAAAAAAAA
AAAAABAAAAAAAABAAAAA
AAAAAABAAAAAABAAAAAA
AAAAAAAAAAAAAAAAAAAA
AAAAAAAAAAAAAAAAAAAA
AAAAAAAAAAAAAAAAAAAA
AAAAAAAAAAAAAAAAAAAA
AAAAAAAAAAAAAABAAAAA
AAAAABAAAAAAAAAAAAAA
AAAAAAAAAAAAAAAAAAAA
BAAAAAAAAAAAAAAAAAAA
AAAAAAAAAAAAABAAAAAA
AAAAAAAAAAAAAAAABABA
AAAAAAAAAAAAAAAAAAAA
AAAAAAAAAAAAAAAAAAAA



In [36]:
cv2.imshow("ty", photo)
cv2.waitKey()
cv2.destroyAllWindows()
cv2.waitKey(1)

-1